# Python 工程化脚本入门

本 notebook 对应 Junior DE checklist 的 Python 代码组织部分：拆函数、`main()`、`if __name__ == "__main__"`、`dataclass`、`logging`。目标是把一段能跑的临时代码整理成别人也能运行、排查和维护的小脚本。

| 模块 | 目标 |
|---|---|
| 函数拆分 | 把 extract / transform / load 分开 |
| main 入口 | 让脚本可以被命令行运行，也可以被导入测试 |
| dataclass | 表达简单配置或业务对象 |
| logging | 记录输入、输出、耗时、异常上下文 |
| 最小 pipeline | 读 CSV、清洗、汇总、写 CSV |


---
## 1. 从长脚本拆成函数

一个 junior 级别的数据脚本至少要能拆成三段：

- `extract`：读取输入
- `transform`：做清洗、过滤、聚合
- `load`：写出结果

核心规则：transform 尽量只处理数据，不直接读写文件。这样后续写单元测试会简单很多。


In [ ]:
def transform_orders(records):
    summary = {}

    for record in records:
        if record["status"] != "paid":
            continue

        customer_id = record["customer_id"]
        amount = float(record["amount"])

        if customer_id not in summary:
            summary[customer_id] = {"customer_id": customer_id, "order_count": 0, "total_amount": 0.0}

        summary[customer_id]["order_count"] += 1
        summary[customer_id]["total_amount"] += amount

    return list(summary.values())


sample_orders = [
    {"order_id": "1", "customer_id": "C001", "amount": "120.5", "status": "paid"},
    {"order_id": "2", "customer_id": "C002", "amount": "50", "status": "cancelled"},
    {"order_id": "3", "customer_id": "C001", "amount": "89.5", "status": "paid"},
]

transform_orders(sample_orders)


---
## 2. main() 与脚本入口

`if __name__ == "__main__"` 的意义是：这个文件被直接运行时执行 `main()`；被别的测试或脚本导入时，不自动跑完整 pipeline。

真实项目里建议把参数放进 `main()`，不要把路径散落在全局代码里。


In [ ]:
from pathlib import Path
import csv


def read_orders_csv(input_path):
    with Path(input_path).open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def write_summary_csv(output_path, records):
    fieldnames = ["customer_id", "order_count", "total_amount"]
    with Path(output_path).open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def run_pipeline(input_path, output_path):
    raw_orders = read_orders_csv(input_path)
    summary = transform_orders(raw_orders)
    write_summary_csv(output_path, summary)
    return len(raw_orders), len(summary)


---
## 3. class / dataclass 基础

Junior 阶段不需要过度设计类。优先使用 `dataclass` 表达配置或简单业务对象，例如输入路径、输出路径、运行日期。


In [ ]:
from dataclasses import dataclass


@dataclass
class PipelineConfig:
    input_path: Path
    output_path: Path
    run_date: str


config = PipelineConfig(
    input_path=Path("tmp_engineering/orders.csv"),
    output_path=Path("tmp_engineering/customer_summary.csv"),
    run_date="2026-01-01",
)

config


---
## 4. 用 logging 替代大量 print

`print` 适合临时调试，`logging` 适合脚本运行记录。数据 pipeline 至少记录：输入路径、输出路径、输入行数、输出行数、运行耗时、失败原因。


In [ ]:
import logging
import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s - %(message)s",
)
logger = logging.getLogger("junior_de.orders")


def run_pipeline_with_logging(config):
    start = time.perf_counter()
    logger.info("pipeline started input=%s output=%s run_date=%s", config.input_path, config.output_path, config.run_date)

    try:
        input_rows, output_rows = run_pipeline(config.input_path, config.output_path)
    except Exception:
        logger.exception("pipeline failed input=%s", config.input_path)
        raise

    elapsed = time.perf_counter() - start
    logger.info("pipeline finished input_rows=%s output_rows=%s elapsed_sec=%.3f", input_rows, output_rows, elapsed)
    return input_rows, output_rows


---
## 5. 最小可运行 pipeline

下面的代码在 notebook 中直接构造一个输入 CSV，然后运行 pipeline。真实项目里可以把这些函数放进 `src/` 或脚本文件，再用命令行参数传入路径。


In [ ]:
work_dir = Path("tmp_engineering")
work_dir.mkdir(exist_ok=True)

input_path = work_dir / "orders.csv"
output_path = work_dir / "customer_summary.csv"

with input_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_id", "customer_id", "amount", "status"])
    writer.writeheader()
    writer.writerows(sample_orders)

config = PipelineConfig(input_path=input_path, output_path=output_path, run_date="2026-01-01")
run_pipeline_with_logging(config)

read_orders_csv(output_path)


---
## 6. 脚本文件模板

如果把上面的逻辑整理成 `.py` 文件，结构可以长这样：

```python
def main():
    config = PipelineConfig(
        input_path=Path("data/orders.csv"),
        output_path=Path("data/customer_summary.csv"),
        run_date="2026-01-01",
    )
    run_pipeline_with_logging(config)


if __name__ == "__main__":
    main()
```

后续升级方向：把路径改成命令行参数，给 `transform_orders` 写 pytest 单元测试，再把输入输出格式扩展到 Parquet 或数据库。


---
## 阶段验收

完成后你应该能独立做到：

1. 把一段长脚本拆成 `read` / `transform` / `write` / `run_pipeline`
2. 使用 `main()` 和 `if __name__ == "__main__"` 组织入口
3. 用 `dataclass` 表达简单配置
4. 用 `logging.info` 记录正常流程，用 `logging.exception` 记录异常上下文
5. 写出一个可重复运行的小型 batch pipeline

练习项目：把 `01_基础语法文件异常.ipynb` 的订单 JSON 汇总练习改写成一个 `.py` 脚本，要求包含 `main()`、日志、异常处理和至少一个可测试的 transform 函数。
